In [6]:
documents = [
    "Python is widely used for backend web development.",
    "Machine learning allows computers to learn patterns from data.",
    "FastAPI is a modern Python framework for building APIs.",
    "Neural networks are inspired by the human brain.",
    "Pandas is useful for data analysis and manipulation.",
    "Semantic search helps find results based on meaning, not exact words.",
    "Embeddings convert text into numerical vectors.",
    "Flask is a lightweight framework for web applications.",
    "Deep learning is a branch of machine learning.",
    "APIs allow different software systems to communicate."
]

In [7]:
def keyword_search(query):
    query_words = query.lower().split()
    results = []

    for doc in documents:
        doc_lower = doc.lower()
        score = 0

        for word in query_words:
            if word in doc_lower:
                score += 1

        if score > 0:
            results.append((doc, score))

    results.sort(key=lambda x: x[1], reverse=True)
    return results

In [8]:
query = "python api"
results = keyword_search(query)

print("Keyword Search Results:\n")
if results:
    for doc, score in results:
        print(f"{doc}  --> score: {score}")
else:
    print("No results found.")

Keyword Search Results:

FastAPI is a modern Python framework for building APIs.  --> score: 2
Python is widely used for backend web development.  --> score: 1
APIs allow different software systems to communicate.  --> score: 1


In [9]:
query = "how computers learn"
results = keyword_search(query)

print("Keyword Search Results:\n")
if results:
    for doc, score in results:
        print(f"{doc}  --> score: {score}")
else:
    print("No results found.")

Keyword Search Results:

Machine learning allows computers to learn patterns from data.  --> score: 2
Deep learning is a branch of machine learning.  --> score: 1


In [10]:
query = "meaning of text"
results = keyword_search(query)

print("Keyword Search Results:\n")
if results:
    for doc, score in results:
        print(f"{doc}  --> score: {score}")
else:
    print("No results found.")

Keyword Search Results:

Semantic search helps find results based on meaning, not exact words.  --> score: 1
Embeddings convert text into numerical vectors.  --> score: 1
Deep learning is a branch of machine learning.  --> score: 1
APIs allow different software systems to communicate.  --> score: 1


In [11]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [12]:
model = SentenceTransformer('all-MiniLM-L6-v2') 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
doc_embeddings = model.encode(documents) 

In [14]:
def semantic_search(query):
    query_embedding = model.encode(query)

    similarities = []

    for i, doc_embedding in enumerate(doc_embeddings):
        similarity = np.dot(query_embedding, doc_embedding) / (
            np.linalg.norm(query_embedding) * np.linalg.norm(doc_embedding)
        )
        similarities.append((documents[i], similarity))

    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities

In [15]:
query = "how computers learn"
results = semantic_search(query)

print("Semantic Search Results:\n")
for doc, score in results[:3]:
    print(f"{doc} --> similarity: {score:.4f}")

Semantic Search Results:

Machine learning allows computers to learn patterns from data. --> similarity: 0.6302
Neural networks are inspired by the human brain. --> similarity: 0.4491
Deep learning is a branch of machine learning. --> similarity: 0.4223


In [16]:
query = "how computers learn"
results = keyword_search(query)

print("Keyword Search:\n")
for doc, score in results:
    print(f"{doc} --> score: {score}")

Keyword Search:

Machine learning allows computers to learn patterns from data. --> score: 2
Deep learning is a branch of machine learning. --> score: 1


In [17]:
query = "how computers learn"
results = semantic_search(query)

print("\nSemantic Search:\n")
for doc, score in results[:3]:
    print(f"{doc} --> similarity: {score:.4f}")


Semantic Search:

Machine learning allows computers to learn patterns from data. --> similarity: 0.6302
Neural networks are inspired by the human brain. --> similarity: 0.4491
Deep learning is a branch of machine learning. --> similarity: 0.4223


In [18]:
def answer_query(query):
    results = semantic_search(query)
    best_match = results[0][0]

    return f"Based on what I found:\n{best_match}"

In [19]:
query = "how do computers learn?"
print(answer_query(query))

Based on what I found:
Machine learning allows computers to learn patterns from data.


In [20]:
from transformers import pipeline 

In [21]:
generator = pipeline("text-generation", model="distilgpt2")
print("generator ready")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generator ready


In [22]:
def rag_answer_local(query):
    results = semantic_search(query)
    top_docs = [doc for doc, _ in results[:3]]
    context = "\n".join(top_docs)

    prompt = f"""Use the context below to answer the question as clearly as possible.

Context:
{context}

Question: {query}

Answer:"""

    output = generator(
        prompt,
        max_new_tokens=80,
        do_sample=True,
        truncation=True
    )

    return output[0]["generated_text"]

In [23]:
query = "how do computers learn?"
print(rag_answer_local(query))

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Use the context below to answer the question as clearly as possible.

Context:
Machine learning allows computers to learn patterns from data.
Neural networks are inspired by the human brain.
Deep learning is a branch of machine learning.

Question: how do computers learn?

Answer: Deep learning is simple to understand and is not a fully integrated way to learn. It has been shown to be effective in learning the human brain.
The main idea behind deep learning is that it can provide a better understanding of what the human brain is doing.
Neural networks are similar to those in the brain, because they are able to pick up information from a neuron, which is a brain
